# 06 — Bypass Triage & Shelf-Life Rescans

**Purpose.** Profile confirmed bypasses by dangerous callable, then measure retention against historical scanner versions (H3). Requires the historic images built (see README).

**Inputs / outputs**
- `data/regenbench_campaign.db` (bypasses)
- historic scanner images (picklescan 1.0.4/1.0.3, modelscan 0.8.7/0.8.6, fickling 0.1.11/0.1.10)

**Outputs**
- `docs/triage-report.md`
- `data/shelf_life.db` (retention per version)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/triage_bypasses.py"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
show("docs/triage-report.md", max_lines=60)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "-c", "from pipeline.shelf_life import register_bypasses_from_campaign_db; register_bypasses_from_campaign_db('data/regenbench_campaign.db')"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
# example single rescan (repeat per historical image; see README step 5)
run(["python3", "scripts/shelf_life_rescan.py", "--db", "data/regenbench_campaign.db",
     "--image", "picklescan=regenbench/picklescan:1.0.4", "--scanners", "picklescan", "--backend", "docker"], check=False)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
print("retention by version:")
print(sqlite("SELECT new_version, total, retained, printf('%.1f%%', retention_rate*100) FROM (SELECT new_version, COUNT(*) total, SUM(evasion_retained) retained, AVG(evasion_retained) retention_rate FROM rescans GROUP BY new_version);", "data/shelf_life.db"))